In [3]:
import pandas as pd

In [4]:
ww = pd.read_csv("../../data/clean/wastewater_treatment.csv")

In [5]:
# Create a table showing number of countries meeting population and treatment thresholds
pop_thresholds = [10, 20, 50]  # in millions
wwt_thresholds = [25, 30, 40, 50]  # in percent

# Convert population to millions for easier comparison
ww['pop_millions'] = ww['pop'] / 1_000_000

# Create results table
results = []
for pop_thresh in pop_thresholds:
    row = {'pop_threshold_millions': pop_thresh}
    for wwt_thresh in wwt_thresholds:
        # Count countries meeting both criteria
        count = len(ww[(ww['pop_millions'] > pop_thresh) & 
                       (ww['wwt_percent'] > wwt_thresh)])
        row[f'wwt>{wwt_thresh}%'] = count
    results.append(row)

# Convert to DataFrame for nice display
threshold_table = pd.DataFrame(results)
threshold_table = threshold_table.set_index('pop_threshold_millions')
print("Number of countries with population > threshold (millions) AND wastewater treatment > threshold (%):")
print(threshold_table)


Number of countries with population > threshold (millions) AND wastewater treatment > threshold (%):
                        wwt>25%  wwt>30%  wwt>40%  wwt>50%
pop_threshold_millions                                    
10                           36       34       30       23
20                           24       22       20       14
50                           13       12       12        8


In [6]:
# Make it really simple
# Just assume some regional integration programme.
# Filter countries with >30% wastewater treatment
ww_filtered = ww[ww['wwt_percent'] > 30].copy()

# Group by region and sum population
regional_pop = ww_filtered.groupby('region')['pop'].sum().sort_values(ascending=False)

# Convert to millions for readability
regional_pop_millions = regional_pop / 1_000_000

print("Total population (millions) by region for countries with >30% wastewater treatment:")
print(regional_pop_millions)

# Calculate population weighted by wastewater treatment rates
ww_filtered['pop_weighted'] = ww_filtered['pop'] * ww_filtered['wwt_percent'] / 100
regional_pop_weighted = ww_filtered.groupby('region')['pop_weighted'].sum().sort_values(ascending=False)
regional_pop_weighted_millions = regional_pop_weighted / 1_000_000

print("\nPopulation weighted by treatment rate (millions) by region:")
print(regional_pop_weighted_millions)

# Extract US population specifically
us_weighted_pop = ww_filtered[ww_filtered['country'] == 'United States']['pop_weighted'].values[0] if 'United States' in ww['country'].values else None
us_weighted_pop_millions = us_weighted_pop / 1_000_000 if us_weighted_pop is not None else None
print(f"\nUS treatment population: {us_weighted_pop_millions:.2f} million")


Total population (millions) by region for countries with >30% wastewater treatment:
region
East Asia & Pacific              1577.288292
Western Europe                    432.970755
Latin America & Caribbean         427.243501
North America                     381.464223
Eastern Europe & Central Asia     321.732370
Middle East & North Africa        302.524906
Sub-Saharan Africa                 72.326927
Southern Asia                       0.527799
Name: pop, dtype: float64

Population weighted by treatment rate (millions) by region:
region
East Asia & Pacific              815.092158
Western Europe                   366.871781
North America                    258.391763
Latin America & Caribbean        190.245261
Middle East & North Africa       171.133772
Eastern Europe & Central Asia    169.719420
Sub-Saharan Africa                30.052608
Southern Asia                      0.195708
Name: pop_weighted, dtype: float64

US treatment population: 230.26 million


In [9]:
import math

# Calculate for divisors 5 and 13
divisors = [5, 13, 2]
results = []

for divisor in divisors:
    # Calculate US treatment pop / divisor
    ratio = us_weighted_pop_millions / divisor
    
    # Multiply by population weighted by treatment rate for each region
    scaled_values = regional_pop_weighted_millions / ratio
    
    # Round upward
    rounded_values = scaled_values.apply(math.ceil)
    
    # Sum the results
    total = rounded_values.sum()
    
    print(f"\nUS treatment pop / {divisor} = {ratio:.2f}")
    print(f"Scaled and rounded values by region:")
    print(rounded_values)
    print(f"Sum: {total}")
    
    results.append({'divisor': divisor, 'sum': total})

print("\n" + "="*50)
print("Summary:")
for r in results:
    print(f"Divisor {r['divisor']}: Sum = {r['sum']}")



US treatment pop / 5 = 46.05
Scaled and rounded values by region:
region
East Asia & Pacific              18
Western Europe                    8
North America                     6
Latin America & Caribbean         5
Middle East & North Africa        4
Eastern Europe & Central Asia     4
Sub-Saharan Africa                1
Southern Asia                     1
Name: pop_weighted, dtype: int64
Sum: 47

US treatment pop / 13 = 17.71
Scaled and rounded values by region:
region
East Asia & Pacific              47
Western Europe                   21
North America                    15
Latin America & Caribbean        11
Middle East & North Africa       10
Eastern Europe & Central Asia    10
Sub-Saharan Africa                2
Southern Asia                     1
Name: pop_weighted, dtype: int64
Sum: 117

US treatment pop / 2 = 115.13
Scaled and rounded values by region:
region
East Asia & Pacific              8
Western Europe                   4
North America                    3
Latin Americ